In [25]:
from pathlib import Path
import mne
import numpy as np
import json
import csv

PROJECT_ROOT = Path.cwd().resolve()

# Nach oben gehen, bis das Datenverzeichnis gefunden wird
while not (PROJECT_ROOT / "data" / "raw" / "ds005363").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Projektverzeichnis nicht gefunden.")

    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "ds005363"

print("Arbeitsverzeichnis:", Path.cwd())
print("Datenverzeichnis:", DATA_ROOT)

def read_tsv(path: Path) -> list[dict[str, str]]:
    """Liest eine TSV-Datei robust gegen ein UTF-8-BOM ein."""

    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file, delimiter="\t"))

def first_matching(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"Keine Datei mit Muster {pattern!r} in {directory}")
    return matches[0]

def subject_directory(data_root: Path, subject: str) -> Path:
    subject_label = subject if subject.startswith("sub-") else f"sub-{subject}"
    directory = data_root / subject_label / "ses-01" / "eeg"
    if not directory.is_dir():
        raise FileNotFoundError(f"EEG-Verzeichnis nicht gefunden: {directory}")
    return directory

def read_data(subject: str, duration: float, data_root: Path, output_root: Path): 
    eeg_dir = subject_directory(data_root, subject)

    vhdr_path = first_matching(eeg_dir, "*_eeg.vhdr")
    json_path = first_matching(eeg_dir, "*_eeg.json")
    channels_path = first_matching(eeg_dir, "*_channels.tsv")
    events_path = first_matching(eeg_dir, "*_events.tsv")

    with json_path.open("r", encoding="utf-8") as file:
        metadata = json.load(file)
    channels = read_tsv(channels_path)
    events = read_tsv(events_path)

    print(f"Lese BrainVision-Aufnahme ein: {vhdr_path}")
    raw = mne.io.read_raw_brainvision(vhdr_path, preload=False, verbose="ERROR")
    eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])
    if len(eeg_picks) == 0:
        raise RuntimeError("Die Aufnahme enthält keine als EEG markierten Kanäle.")

    return raw, channels, events



Arbeitsverzeichnis: /Users/benediktadler/Documents/Uni Projekte/bachelor-thesis/notebooks/BunDLe-Net/notebooks
Datenverzeichnis: /Users/benediktadler/Documents/Uni Projekte/bachelor-thesis/data/raw/ds005363


In [ ]:
try:
    raw, channels, events = read_data(
        subject="101",
        duration=60.0,
        data_root=DATA_ROOT,
        output_root=PROJECT_ROOT / "results",
    )

except FileNotFoundError as exc:
    raise SystemExit(f"Datei nicht verfügbar: {exc}") from exc
except RuntimeError as exc:
    raise SystemExit(f"Analyse konnte nicht durchgeführt werden: {exc}") from exc


# EEG-Kanäle auswählen
eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])

# Messwerte: (Kanäle, Zeitpunkte)
neuron_traces = raw.get_data(picks=eeg_picks)

# Kanalnamen in derselben Reihenfolge wie die Messwerte
channel_names = np.array(
    [raw.ch_names[i] for i in eeg_picks],
    dtype=object
)

# Abtastrate der Aufnahme: bei dir 1000 Hz
fps = raw.info["sfreq"]
state_names = [
    "keine Kategorie",
    "Gesicht",
    "Tier",
    "Ort",
    "Objekt",
    "Catch",
]

stimulus_to_state = {}

for code in range(1, 17):
    stimulus_to_state[code] = 1  # Gesicht

for code in range(17, 33):
    stimulus_to_state[code] = 2  # Tier

for code in range(33, 49):
    stimulus_to_state[code] = 3  # Ort

for code in range(49, 65):
    stimulus_to_state[code] = 4  # Objekt

for code in range(65, 70):
    stimulus_to_state[code] = 5
print("--------------------------------------- STIMULUS TO STATE ARRAY ---------------------------------------")
print(stimulus_to_state)
print("")
print("")

event = events[4] 

sample = int(event["sample"])
messwerte = neuron_traces[:, sample]

print("--------------------------------------- Neuron Traces ---------------------------------------")
print(neuron_traces[:,:])
print("")
print(neuron_traces[[0],:])
print("")
print(neuron_traces.shape)

print("--------------------------------------- EVENT NR. 4 ---------------------------------------")
print(event)
print("")
print("")
print("--------------------------------------- MESSWERTE FÜR SAMPLE VON EVENT NR. 4 ---------------------------------------")
print(messwerte)
print("")
print("")
print("--------------------------------------- FORM VON MESSWERTE ---------------------------------------")
print(messwerte.shape)
print("")
print("")
print("--------------------------------------- FORM VON NEURON TRACES ---------------------------------------")
print(neuron_traces.shape)
print("")
print("")
# print("--------------------------------------- FORM VON B ---------------------------------------")
# B = np.zeros(neuron_traces.shape[1], dtype=int)
# print(B.shape)

Lese BrainVision-Aufnahme ein: /Users/benediktadler/Documents/Uni Projekte/bachelor-thesis/data/raw/ds005363/sub-101/ses-01/eeg/sub-101_ses-01_task-objrecog_eeg.vhdr
--------------------------------------- STIMULUS TO STATE ARRAY ---------------------------------------
{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1, 14: 1, 15: 1, 16: 1, 17: 2, 18: 2, 19: 2, 20: 2, 21: 2, 22: 2, 23: 2, 24: 2, 25: 2, 26: 2, 27: 2, 28: 2, 29: 2, 30: 2, 31: 2, 32: 2, 33: 3, 34: 3, 35: 3, 36: 3, 37: 3, 38: 3, 39: 3, 40: 3, 41: 3, 42: 3, 43: 3, 44: 3, 45: 3, 46: 3, 47: 3, 48: 3, 49: 4, 50: 4, 51: 4, 52: 4, 53: 4, 54: 4, 55: 4, 56: 4, 57: 4, 58: 4, 59: 4, 60: 4, 61: 4, 62: 4, 63: 4, 64: 4, 65: 5, 66: 5, 67: 5, 68: 5, 69: 5}


--------------------------------------- Neuron Traces ---------------------------------------
[[-5.050e-05 -2.600e-05 -4.000e-06 ...  6.900e-05  6.700e-05  6.550e-05]
 [-5.250e-05 -5.500e-05 -5.750e-05 ...  1.175e-04  1.165e-04  1.155e-04]
 [ 1.000e-05 

In [126]:
def extract_s(trial_type):
    if not trial_type.startswith("Stimulus/S"):
        return None

    number = trial_type.removeprefix("Stimulus/S").strip()
    return int(number)

labels = []
B = np.zeros(neuron_traces.shape[1], dtype=int)
for event in events:
    code = extract_s(event["trial_type"])

    if code in stimulus_to_state:
        label = stimulus_to_state[code]
    elif code == 199 or code in range(101, 109):
        label = 0  
    else:
        continue 

    sample = int(event["sample"])
    labels.append((sample, label))

l = len(labels)

for e in range(l): 
    current_sample, label = labels[e]
    if e+1 < l:
        next_sample = labels[e+1][0]
    else:
        next_sample = l
    
    B[current_sample:next_sample] = label

print(B[1461512:1462449])
    


BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "gamma": (30.0, 45.0),
}






[4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 